# Data Modeling Task
The goal of this task is to examine the master census metrics JSON file for the CCSVI Dashboard and understand how to transition the existing data into a format usable by an LLM.

### Updates
- master JSON file: `public/data/metrics/census_metrics_by_block_group.json`.
- flattened the nested JSON into a long-form Pandas DataFrame.

# Inspecting the Master JSON

In [24]:
import json
import pandas as pd

In [25]:
# Loading the master .json file
master_file = "../public/data/metrics/census_metrics_by_block_group.json"
with open(master_file, "r") as f:
    data = json.load(f)

In [26]:
# Examining structure
sample_geoid = list(data.keys())[0]
sample = data[sample_geoid]

print("Sample GEOID:", sample_geoid)
print("\nTop-level keys:")
print(sample.keys())

print("\nGeographic Info:")
for key in ["type", "block_group", "census_tract", "county", "state", "population"]:
    print(f"{key}: {sample.get(key)}")

print("\nMetrics datasets:")
print(sample["metrics"].keys())

first_dataset = list(sample["metrics"].keys())[0]
print("\nSample metric names:")
print(list(sample["metrics"][first_dataset].keys())[:10])

Sample GEOID: 5003

Top-level keys:
dict_keys(['type', 'name', 'block_group', 'census_tract', 'county', 'state', 'population', 'metrics'])

Geographic Info:
type: hawaiian_homeland
block_group: None
census_tract: None
county: None
state: None
population: 257

Metrics datasets:
dict_keys(['2022_census_hawaiian_homelands'])

Sample metric names:
['Total Population Under 5', 'Total Population Under 18', 'Total Population Over 65', 'Total population SEX Male', 'Total population SEX Female', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race White', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Black or African American', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race American Indian and Alaska Native', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Asian', 'Total population RACE AND HISPANIC OR LATINO ORIGIN One race Native Hawaiian and Other Pacific Islander']


In [27]:
# Pivot into long data frame
rows = []

for geoid, area_data in data.items():
    base_info = {
        "geoid": geoid,
        "type": area_data.get("type"),
        "name": area_data.get("name"),
        "block_group": area_data.get("block_group"),
        "census_tract": area_data.get("census_tract"),
        "county": area_data.get("county"),
        "state": area_data.get("state"),
        "population": area_data.get("population")
    }
    
    for dataset_name, metrics in area_data.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            rows.append({
                **base_info,
                "dataset": dataset_name,
                "metric": metric_name,
                "absolute": values.get("absolute"),
                "proportion": values.get("proportion")
            })

df_long = pd.DataFrame(rows)
df_long.head()
print("Shape:", df_long.shape)
type_counts = df_long.groupby("type").size().reset_index(name="row_count")
type_counts

Shape: (44520, 12)


,type,row_count
0,block_group,43320
1,hawaiian_homeland,1200


In [28]:
# Compare metrics between block groups and Hawaiian homelands
metrics_block = set(df_long[df_long["type"] == "block_group"]["metric"].unique())
metrics_hh = set(df_long[df_long["type"] == "hawaiian_homeland"]["metric"].unique())

print("Total Block Group metrics:", len(metrics_block))
print("Total Hawaiian Homeland metrics:", len(metrics_hh))

# Metrics shared by both types
shared_metrics = metrics_block.intersection(metrics_hh)
print("\nShared metrics:", len(shared_metrics))
print(sorted(shared_metrics))

# Metrics unique to Block Groups
block_only = metrics_block - metrics_hh
print("\nMetrics only in Block Groups:", len(block_only))
print(sorted(block_only))

# Metrics unique to Hawaiian Homelands
hh_only = metrics_hh - metrics_block
print("\nMetrics only in Hawaiian Homelands:", len(hh_only))
print(sorted(hh_only))

Total Block Group metrics: 40
Total Hawaiian Homeland metrics: 16

Shared metrics: 0
[]

Metrics only in Block Groups: 40
['American Indian and Alaska Native alone', 'Asian alone', 'Asian and Pacific Island languages: Limited English speaking household', 'Black or African American alone', 'Estimate Aggregate number of vehicles available', 'Estimate Total', 'Female', 'Females Over 65', 'Females Under 18', 'Females Under 5', 'In households: Householder: Female: Living alone', 'In households: Householder: Male: Living alone', 'Institutionalized population', 'Institutionalized population: Correctional facilities for adults', 'Institutionalized population: Juvenile facilities', 'Institutionalized population: Nursing facilities/Skilled-nursing facilities', 'Institutionalized population: Other institutional facilities', 'Male', 'Males Over 65', 'Males Under 18', 'Males Under 5', 'Native Hawaiian and Other Pacific Islander alone', 'No Computer', 'No Health Insurance Coverage', 'No Internet acc

## Structural Observations

1. Two geographic types exist:
   - block_group (standard Census hierarchy — nested within census tracts and counties)
   - hawaiian_homeland (a separate geographic designation that does not follow the tract → block group nesting)

2. Metrics are nested by:
   GEOID → dataset → metric_name → {absolute, proportion}

3. Block groups and Hawaiian Homelands do not have the same metric coverage.
   - Block groups contain 40 metrics.
   - Hawaiian Homelands contain 16 metrics.
   - Some indicators (e.g., institutionalized population, internet access, vehicle access) only exist for block groups.

In [29]:
# Exploring a schema design
df_geo = (
    df_long[df_long["type"] == "block_group"]
    [["geoid", "block_group", "census_tract", "county", "state", "population"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_geo.head()
df_geo = pd.DataFrame(rows)

df_geo.head()

,geoid,type,name,block_group,census_tract,county,state,population,dataset,metric,absolute,proportion
0,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total Population Under 5,16.0,0.062257
1,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total Population Under 18,39.0,0.151751
2,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total Population Over 65,21.0,0.081712
3,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total population SEX Male,43.0,0.167315
4,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0,2022_census_hawaiian_homelands,Total population SEX Female,56.0,0.217899


---
# Schema Design

Based on the structure of the master JSON, we can model the data into **3 relational tables**.
This mirrors how the data would be stored in a SQL database (e.g. PostgreSQL + PostGIS).

| Table | Description |
|---|---|
| `geographic_areas` | One row per GEOID — location and population info |
| `datasets` | Reference table for the 14 source metric datasets |
| `vulnerability_metrics` | Long-form fact table — every metric value per area |

> **Note on PostGIS:** A future `geometry` column on `geographic_areas` would store the actual
> polygon shape of each census block group (sourced from GeoJSON/GeoTIFF files).
> This would allow spatial queries like: *which high-poverty block groups intersect a flood hazard zone?*

## Table 1: `geographic_areas`
One row per GEOID. Captures the geographic hierarchy and population for each area.

In [30]:
# Build geographic_areas table
geo_rows = []

for geoid, area in data.items():
    geo_rows.append({
        "geoid":        geoid,
        "type":         area.get("type"),
        "name":         area.get("name"),
        "block_group":  area.get("block_group"),
        "census_tract": area.get("census_tract"),
        "county":       area.get("county"),
        "state":        area.get("state"),
        "population":   area.get("population"),
        # geometry column placeholder — will be populated once GeoJSON/PostGIS is integrated
        # "geometry": None
    })

df_geo = pd.DataFrame(geo_rows)

print("Shape:", df_geo.shape)
print("\nCounts by type:")
print(df_geo["type"].value_counts())
print("\nCounts by county:")
print(df_geo["county"].value_counts())
df_geo.head()

Shape: (1158, 8)

Counts by type:
type
block_group          1083
hawaiian_homeland      75
Name: count, dtype: int64

Counts by county:
county
Honolulu County    774
Hawaii County      147
Maui County        107
Kauai County        53
Kalawao County       2
Name: count, dtype: int64


,geoid,type,name,block_group,census_tract,county,state,population
0,5003,hawaiian_homeland,"Anahola (Agricultural) Hawaiian Home Land, HI",None,None,None,None,257.0
1,5004,hawaiian_homeland,"Anahola (Residential) Hawaiian Home Land, HI",None,None,None,None,1566.0
2,5008,hawaiian_homeland,"East Kapolei Hawaiian Home Land, HI",None,None,None,None,0.0
3,5009,hawaiian_homeland,"Haiku Hawaiian Home Land, HI",None,None,None,None,0.0
4,5011,hawaiian_homeland,"Hanapepe Hawaiian Home Land, HI",None,None,None,None,25.0


## Table 2: `datasets`
A reference/lookup table for the 14 source datasets.
All block group datasets come from the ACS (American Community Survey).
The Hawaiian Homeland dataset is a separate 2022 Census designation.

In [31]:
# Build datasets reference table
from collections import defaultdict

dataset_geo_type = defaultdict(set)
for geoid, area in data.items():
    for ds in area.get("metrics", {}).keys():
        dataset_geo_type[ds].add(area.get("type"))

dataset_rows = [
    {"dataset_id": ds, "geo_type": ", ".join(sorted(types))}
    for ds, types in sorted(dataset_geo_type.items())
]

df_datasets = pd.DataFrame(dataset_rows)
print("Total datasets:", len(df_datasets))
df_datasets

Total datasets: 15


,dataset_id,geo_type
0,2022_census_hawaiian_homelands,hawaiian_homeland
1,age_of_structure,block_group
2,aggregate_vehicles,block_group
3,genders,block_group
4,health_insurance,block_group
5,households_w_computer,block_group
6,income_share_of_fpl,block_group
7,internet_subscription,block_group
8,limited_english_speaking,block_group
9,living_arrangements,block_group


## Table 3: `vulnerability_metrics`
The core fact table. One row per (geoid, dataset, metric) combination.
Each metric has an `absolute` count and a `proportion` (share of population).

This is the table the LLM will query most heavily.

In [32]:
# Build vulnerability_metrics table (long-form fact table)
metric_rows = []

for geoid, area in data.items():
    for dataset_id, metrics in area.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            metric_rows.append({
                "geoid":      geoid,
                "dataset_id": dataset_id,
                "metric":     metric_name,
                "absolute":   values.get("absolute"),
                "proportion": values.get("proportion"),
            })

df_metrics = pd.DataFrame(metric_rows)

print("Shape:", df_metrics.shape)
print(f"\nNull absolute values:   {df_metrics['absolute'].isna().sum()}")
print(f"Null proportion values: {df_metrics['proportion'].isna().sum()}")
df_metrics.head(10)

Shape: (44520, 5)

Null absolute values:   2606
Null proportion values: 3944


,geoid,dataset_id,metric,absolute,proportion
0,5003,2022_census_hawaiian_homelands,Total Population Under 5,16.0,0.062257
1,5003,2022_census_hawaiian_homelands,Total Population Under 18,39.0,0.151751
2,5003,2022_census_hawaiian_homelands,Total Population Over 65,21.0,0.081712
3,5003,2022_census_hawaiian_homelands,Total population SEX Male,43.0,0.167315
4,5003,2022_census_hawaiian_homelands,Total population SEX Female,56.0,0.217899
5,5003,2022_census_hawaiian_homelands,Total population RACE AND HISPANIC OR LATINO O...,16.0,0.062257
6,5003,2022_census_hawaiian_homelands,Total population RACE AND HISPANIC OR LATINO O...,0.0,0.000000
7,5003,2022_census_hawaiian_homelands,Total population RACE AND HISPANIC OR LATINO O...,0.0,0.000000
8,5003,2022_census_hawaiian_homelands,Total population RACE AND HISPANIC OR LATINO O...,1.0,0.003891
9,5003,2022_census_hawaiian_homelands,Total population RACE AND HISPANIC OR LATINO O...,29.0,0.112840


## Schema Validation
Checking referential integrity — every metric row should have a matching GEOID in `geographic_areas`,
and every dataset_id should exist in the `datasets` reference table.

In [33]:
# Validate: all GEOIDs in metrics exist in geographic_areas
geoids_in_geo     = set(df_geo["geoid"])
geoids_in_metrics = set(df_metrics["geoid"])
orphaned_geoids   = geoids_in_metrics - geoids_in_geo

print("=== GEOID Integrity ===")
print(f"GEOIDs in geographic_areas:      {len(geoids_in_geo)}")
print(f"GEOIDs in vulnerability_metrics: {len(geoids_in_metrics)}")
print(f"Orphaned GEOIDs:                 {len(orphaned_geoids)}")

# Validate: all dataset_ids in metrics exist in datasets reference table
ds_in_ref     = set(df_datasets["dataset_id"])
ds_in_metrics = set(df_metrics["dataset_id"])
missing_ds    = ds_in_metrics - ds_in_ref

print("\n=== Dataset ID Integrity ===")
print(f"Dataset IDs in reference table: {len(ds_in_ref)}")
print(f"Dataset IDs in metrics:         {len(ds_in_metrics)}")
print(f"Missing dataset IDs:            {len(missing_ds)}")

=== GEOID Integrity ===
GEOIDs in geographic_areas:      1158
GEOIDs in vulnerability_metrics: 1158
Orphaned GEOIDs:                 0

=== Dataset ID Integrity ===
Dataset IDs in reference table: 15
Dataset IDs in metrics:         15
Missing dataset IDs:            0


## SQL Schema (DDL)
This is what the schema would look like written as SQL — the language used to define
and query a relational database like PostgreSQL.

The commented-out `geometry` column is where **PostGIS** would come in:
storing the actual polygon boundary of each block group so we can run
spatial queries against climate hazard layers (e.g. flood zones, wildfire risk).

In [34]:
sql_schema = """
-- Table 1: Geographic areas (census block groups + Hawaiian homelands)
CREATE TABLE geographic_areas (
    geoid           TEXT PRIMARY KEY,
    type            TEXT,              -- 'block_group' or 'hawaiian_homeland'
    name            TEXT,
    block_group     TEXT,
    census_tract    TEXT,
    county          TEXT,
    state           TEXT,
    population      INTEGER
    -- geometry    GEOMETRY(MULTIPOLYGON, 4326)  -- PostGIS: add once GeoJSON is integrated
);

-- Table 2: Dataset reference table
CREATE TABLE datasets (
    dataset_id      TEXT PRIMARY KEY,
    geo_type        TEXT               -- 'block_group' or 'hawaiian_homeland'
);

-- Table 3: Vulnerability metrics (fact table)
CREATE TABLE vulnerability_metrics (
    geoid           TEXT REFERENCES geographic_areas(geoid),
    dataset_id      TEXT REFERENCES datasets(dataset_id),
    metric          TEXT,
    absolute        FLOAT,
    proportion      FLOAT,
    PRIMARY KEY (geoid, dataset_id, metric)
);

-- Example query: top 10 block groups by uninsured rate
-- SELECT g.geoid, g.county, g.population, m.proportion
-- FROM vulnerability_metrics m
-- JOIN geographic_areas g ON m.geoid = g.geoid
-- WHERE m.metric = 'No Health Insurance Coverage'
-- ORDER BY m.proportion DESC
-- LIMIT 10;
"""

print(sql_schema)


-- Table 1: Geographic areas (census block groups + Hawaiian homelands)
CREATE TABLE geographic_areas (
    geoid           TEXT PRIMARY KEY,
    type            TEXT,              -- 'block_group' or 'hawaiian_homeland'
    name            TEXT,
    block_group     TEXT,
    census_tract    TEXT,
    county          TEXT,
    state           TEXT,
    population      INTEGER
    -- geometry    GEOMETRY(MULTIPOLYGON, 4326)  -- PostGIS: add once GeoJSON is integrated
);

-- Table 2: Dataset reference table
CREATE TABLE datasets (
    dataset_id      TEXT PRIMARY KEY,
    geo_type        TEXT               -- 'block_group' or 'hawaiian_homeland'
);

-- Table 3: Vulnerability metrics (fact table)
CREATE TABLE vulnerability_metrics (
    geoid           TEXT REFERENCES geographic_areas(geoid),
    dataset_id      TEXT REFERENCES datasets(dataset_id),
    metric          TEXT,
    absolute        FLOAT,
    proportion      FLOAT,
    PRIMARY KEY (geoid, dataset_id, metric)
);

-- Example q

## Schema Observations

1. **1,158 total geographic areas** — 1,083 block groups across 5 Hawaii counties (Hawaii, Honolulu, Kalawao, Kauai, Maui), and 75 Hawaiian homelands.

2. **14 source datasets** — 13 apply only to block groups (ACS data), 1 applies only to Hawaiian homelands (`2022_census_hawaiian_homelands`). There is zero metric overlap between the two geographic types, so the LLM will need to be aware of which type it is querying.

3. **44,520 total metric rows** — with some null values present (~2,600 null absolutes, ~3,900 null proportions). These will need handling before feeding to an LLM (e.g. fill with 0, drop, or flag as missing).

4. **Key vulnerability signals for LLM queries:**
   - `health_insurance` → No Health Insurance Coverage
   - `income_share_of_fpl` → poverty thresholds at 100%, 150%, 200% FPL
   - `limited_english_speaking` → language access barriers
   - `internet_subscription` / `households_w_computer` → digital access gap
   - `tenure` → Renter occupied (housing instability proxy)
   - `population_group_quarters` → institutionalized populations

5. **PostGIS integration (future):** Adding a `geometry` column to `geographic_areas` would enable spatial joins with climate hazard layers (flood zones, wildfire risk, sea level rise, etc.) — the core goal of the CCSVI dashboard pipeline.

## PostgreSQL Start-up

In [35]:
from sqlalchemy import create_engine, text

engine = create_engine("postgresql://postgres@localhost/ccsvi_db")

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected:", result.fetchone())

Connected: ('PostgreSQL 18.3 (Postgres.app) on aarch64-apple-darwin23.6.0, compiled by Apple clang version 15.0.0 (clang-1500.3.9.4), 64-bit',)


In [36]:
# Load DataFrames into PostgreSQL
df_geo.to_sql("geographic_areas", engine, if_exists="replace", index=False)
print("✓ geographic_areas:", len(df_geo), "rows")

df_datasets.to_sql("datasets", engine, if_exists="replace", index=False)
print("✓ datasets:", len(df_datasets), "rows")

df_metrics.to_sql("vulnerability_metrics", engine, if_exists="replace", index=False)
print("✓ vulnerability_metrics:", len(df_metrics), "rows")

✓ geographic_areas: 1158 rows
✓ datasets: 15 rows
✓ vulnerability_metrics: 44520 rows


In [37]:
# Test with a real query
query = """
    SELECT county, COUNT(DISTINCT geoid) AS block_group_count
    FROM geographic_areas
    WHERE type = 'block_group'
    GROUP BY county
    ORDER BY block_group_count DESC;
"""
pd.read_sql(query, engine)

,county,block_group_count
0,Honolulu County,774
1,Hawaii County,147
2,Maui County,107
3,Kauai County,53
4,Kalawao County,2


In [38]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT postgis_version();"))
    print("PostGIS version:", result.fetchone())

PostGIS version: ('3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1',)


## Adding geometry column

In [ ]:
import json
from sqlalchemy import text

# Census Blocks Geometry
geojson_path = "../public/data/2020_Census_Block_Groups_Stripped.geojson"

with open(geojson_path, "r") as f:
    bg_geojson = json.load(f)

with engine.connect() as conn:
    conn.execute(text("ALTER TABLE geographic_areas ADD COLUMN IF NOT EXISTS geom geometry(Geometry, 4326);"))
    conn.commit()

    updated = 0
    skipped = 0
    
    for feature in bg_geojson["features"]:
        geoid = feature["properties"].get("geoid20")
        geometry_json = json.dumps(feature["geometry"])
        
        result = conn.execute(
            text("""
                UPDATE geographic_areas 
                SET geom = ST_SetSRID(ST_GeomFromGeoJSON(:geom_json), 4326)
                WHERE geoid = :geoid
            """), 
            {"geom_json": geometry_json, "geoid": str(geoid)}
        )
        
        if result.rowcount > 0:
            updated += 1
        else:
            skipped += 1
            
    conn.commit()

In [66]:
# Hawaiian Homelands Geometry
hhl_path = "../public/data/Census_Hawaiian_Homelands_hhl10_Stripped.geojson" 

with open(hhl_path, "r") as f:
    hhl_geojson = json.load(f)

with engine.connect() as conn:
    hhl_updated = 0
    hhl_skipped = 0
    
    for feature in hhl_geojson["features"]:
        # Use the key we just verified: 'GEOID10'
        geoid = feature["properties"].get("GEOID10")
        geometry_json = json.dumps(feature["geometry"])
        
        # Update the row where the geoid matches
        result = conn.execute(
            text("""
                UPDATE geographic_areas 
                SET geom = ST_SetSRID(ST_GeomFromGeoJSON(:geom_json), 4326)
                WHERE geoid = :geoid
            """), 
            {"geom_json": geometry_json, "geoid": str(geoid)}
        )
        
        if result.rowcount > 0:
            hhl_updated += 1
        else:
            hhl_skipped += 1
            
    conn.commit()

print(f"Updated {hhl_updated} Hawaiian Home Lands.")
print(f"Skipped {hhl_skipped} (IDs that didn't exist in your table).")

Updated 73 Hawaiian Home Lands.
Skipped 2 (IDs that didn't exist in your table).


In [69]:
# Checks
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            COUNT(geom) as spatial_rows,
            (COUNT(geom)::float / NULLIF(COUNT(*), 0)::float) * 100 as percentage_complete
        FROM geographic_areas;
    """)).fetchone()
    
    # This "unpacks" the row into three separate variables
    total, spatial, percent = result
    
    print(f"Total Database Rows: {total}")
    print(f"Rows with Geometry: {spatial}")

Total Database Rows: 1158
Rows with Geometry: 1129
